In [1]:
!pip install selenium
!pip install webdriver-manager
!pip install beautifulsoup4


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [3]:
def get_only_review_count(text):
    """Extract review count from text like '573 Ratings&40 Reviews' or '(35,184)'"""
    match = re.search(r'([\d,]+)\s*Ratings?', text, re.IGNORECASE)
    if match:
        return match.group(1).replace(',', '')
    text = text.replace('(', '').replace(')', '')
    match = re.search(r'([\d,]+)', text)
    if match:
        return match.group(1).replace(',', '')
    return "0"

def get_product_name(container):
    """Try multiple selectors to get product name"""
    name_selectors = [
        ('a.pIpigb', 'title'),
        ('a.atJtCj', 'title'),
        ('div.RG5Slk', 'text'),
        ('a.wjcEIp', 'title'),
        ('div.KzDlHZ', 'text'),
        ('a.CGtC98', 'title'),
    ]
    for sel, attr in name_selectors:
        el = container.select_one(sel)
        if el:
            name = el.get('title', '') if attr == 'title' else el.get_text(strip=True)
            if name:
                return name
    return None

def scrape_category(base_url, category, num_pages=5):
    """Scrape products from a given Flipkart search URL"""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    data = []

    try:
        for page in range(1, num_pages + 1):
            url = f"{base_url}&page={page}"
            print(f"Scraping {category} - page {page}")
            driver.get(url)
            time.sleep(5)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            product_containers = soup.select("div[data-id]")

            for container in product_containers:
                name = get_product_name(container)
                if not name:
                    continue

                price_el = container.select_one("div.hZ3P6w")
                price = price_el.text.strip() if price_el else ""

                rating_el = container.select_one("div.MKiFS6")
                rating = rating_el.text.strip() if rating_el else ""

                rating_count_el = container.select_one("span.PvbNMB")
                rating_count_text = rating_count_el.text.strip() if rating_count_el else ""
                no_of_ratings = get_only_review_count(rating_count_text)

                if name:
                    data.append({
                        "Product Name":  name,
                        "Category":      category,
                        "Price":         price,
                        "Rating":        rating,
                        "Reviews_Text":  rating_count_text,
                        "No_of_Reviews": no_of_ratings
                    })
            time.sleep(2)
    finally:
        driver.quit()

    return data


In [4]:
BASE_URL = "https://www.flipkart.com/search?q=tyres&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"
CATEGORY = "Tyres"

data_tyres = scrape_category(BASE_URL, CATEGORY)

df_tyres = pd.DataFrame(data_tyres)
df_tyres.to_csv("flipkart_tyres.csv", index=False)
print(f"\nData saved to flipkart_tyres.csv")
print(f"Scraping completed successfully! Total: {len(data_tyres)} products")


Scraping Tyres - page 1
Scraping Tyres - page 2
Scraping Tyres - page 3
Scraping Tyres - page 4
Scraping Tyres - page 5

Data saved to flipkart_tyres.csv
Scraping completed successfully! Total: 200 products


In [5]:
df_tyres = pd.read_csv("flipkart_tyres.csv")
df_tyres


,Product Name,Category,Price,Rating,Reviews_Text,No_of_Reviews
0,JK Taximax 4 Wheeler Tyre,Tyres,"₹3,099",4.2,(6),6
1,JK TYRE UX Royale 4 Wheeler Tyre,Tyres,"₹4,183",3.9,(55),55
2,TVS Eurogrip ATT 52 2.75 - 18 42 P Front Two W...,Tyres,₹884,4.2,(734),734
3,CEAT 105918 Milaze TL 53J SW 90/100-10 Front &...,Tyres,"₹1,082",4.3,"(21,317)",21317
4,MRF Nylogrip FE 90/100 R10 Front & Rear Two Wh...,Tyres,"₹1,215",4.3,(370),370
...,...,...,...,...,...,...
195,MRF 90/80-17 90/80-17 Front Two Wheeler Tyre,Tyres,"₹2,199",4.2,(119),119
196,JK TYRE Taximax 85 S 4 Wheeler Tyre,Tyres,"₹2,957",4.3,"(1,047)",1047
197,Goodyear Kelly KELLY VFM3 4 Wheeler Tyre,Tyres,"₹2,174",3.9,(84),84
198,MRF Zapper S 130/70-17 Rear Two Wheeler Tyre,Tyres,"₹2,748",4.2,(160),160


In [6]:
BASE_URL = "https://www.flipkart.com/search?q=brake+pads&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"
CATEGORY = "Brake"

data_brake = scrape_category(BASE_URL, CATEGORY)

df_brake = pd.DataFrame(data_brake)
df_brake.to_csv("flipkart_brake.csv", index=False)
print(f"\nData saved to flipkart_brake.csv")
print(f"Scraping completed successfully! Total: {len(data_brake)} products")


Scraping Brake - page 1
Scraping Brake - page 2
Scraping Brake - page 3
Scraping Brake - page 4
Scraping Brake - page 5

Data saved to flipkart_brake.csv
Scraping completed successfully! Total: 200 products


In [7]:
df_brake = pd.read_csv("flipkart_brake.csv")
df_brake


,Product Name,Category,Price,Rating,Reviews_Text,No_of_Reviews
0,Online Expert Mountain Bikes Cycling Road Bicy...,Brake,₹250,NaN,NaN,0
1,TRP Traders REAR DISC PAD COMPATIBLE FOR EEVE ...,Brake,₹160,4.2,(204),204
2,Siano 2 Pair Bicycle Disc Braking Pads for MTB...,Brake,₹73,4.1,(231),231
3,LORWADIYA Bicycle Disc Braking Pads 5pair For ...,Brake,₹170,4.1,(217),217
4,Gymisa icycle Disc Braking Pads For MTB Mounta...,Brake,₹121,4.1,(54),54
...,...,...,...,...,...,...
195,flea Height Increase Insole 3-Layer Air Cushio...,Brake,₹313,3.6,(12),12
196,UNO MINDA Brake Lining for Mahindra Scorpio BL...,Brake,₹585,NaN,NaN,0
197,UNO MINDA Brake Lining for Mahindra Scorpio Cr...,Brake,₹818,NaN,NaN,0
198,"Bestor Set of 5 Combo,USB Wired Keyboard,Wire ...",Brake,₹670,3.6,(143),143


In [8]:
BASE_URL = "https://www.flipkart.com/search?q=clutch+plate&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"
CATEGORY = "Clutch"

data_clutch = scrape_category(BASE_URL, CATEGORY)

df_clutch = pd.DataFrame(data_clutch)
df_clutch.to_csv("flipkart_clutch.csv", index=False)
print(f"\nData saved to flipkart_clutch.csv")
print(f"Scraping completed successfully! Total: {len(data_clutch)} products")


Scraping Clutch - page 1
Scraping Clutch - page 2
Scraping Clutch - page 3
Scraping Clutch - page 4
Scraping Clutch - page 5

Data saved to flipkart_clutch.csv
Scraping completed successfully! Total: 200 products


In [9]:
df_clutch = pd.read_csv("flipkart_clutch.csv")
df_clutch


,Product Name,Category,Price,Rating,Reviews_Text,No_of_Reviews
0,jiyamobility SUPER SPLENDOR/GLAMOUR / GLAMOUR ...,Clutch,₹275,NaN,NaN,0
1,KDTRD Clutch Brake Paddle Rubber Cover Pad Cre...,Clutch,₹125,NaN,NaN,0
2,Digital Craft CT 100 Clutch Plate,Clutch,₹282,2.5,(4),4
3,Jagdamba Motorss Clutch Plate Clutch Plate,Clutch,₹171,NaN,NaN,0
4,MOTO PLANET 02CD0080 Clutch Plate,Clutch,"₹1,443",5.0,(5),5
...,...,...,...,...,...,...
195,jop 0899 Brake Caliper,Clutch,₹354,4.3,(110),110
196,GEO Brake Disc Plate Compatible for Bajaj Plat...,Clutch,₹849,2.3,(3),3
197,GEO Front Brake Disc Plate Compatible With KTM...,Clutch,₹778,2.0,(4),4
198,Pa Ktm/Duke Rear Disc Plate Motorbike Brake Disc,Clutch,₹751,4.2,(13),13


In [10]:
BASE_URL = "https://www.flipkart.com/search?q=engine+oil&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"
CATEGORY = "Oil"

data_oil = scrape_category(BASE_URL, CATEGORY)

df_oil = pd.DataFrame(data_oil)
df_oil.to_csv("flipkart_oil.csv", index=False)
print(f"\nData saved to flipkart_oil.csv")
print(f"Scraping completed successfully! Total: {len(data_oil)} products")


Scraping Oil - page 1
Scraping Oil - page 2
Scraping Oil - page 3
Scraping Oil - page 4
Scraping Oil - page 5

Data saved to flipkart_oil.csv
Scraping completed successfully! Total: 200 products


In [11]:
df_oil = pd.read_csv("flipkart_oil.csv")
df_oil


,Product Name,Category,Price,Rating,Reviews_Text,No_of_Reviews
0,MOTUL 4100 Ecomile SAE 5W-30 Technosynthese (S...,Oil,₹419,4.2,(54),54
1,HouseOfCommon ACTION 20W40 1 LTR P1 High Perfo...,Oil,₹215,4.1,"(2,105)",2105
2,ESSON SUPER SYNTH 5W30 3 LTR P1 SUPER SYNTH 5W...,Oil,₹736,4.3,(964),964
3,Shell Advance AX7 10W-40 API SM Synthetic Blen...,Oil,₹340,4.4,"(7,512)",7512
4,AMARON 20W-40 API SL & JASO MA-2 High Performa...,Oil,₹251,4.2,(24),24
...,...,...,...,...,...,...
195,Reddit 20W-40-PREMIUM4T | Smooth Ride|Better M...,Oil,₹226,4.4,(9),9
196,ESSON SUPER SYNTH 5W30 3 LTR P1 SUPER SYNTH 5W...,Oil,₹736,4.3,(964),964
197,PL SUPER PALCO 2T POWER API TC 2 Stroke Air Co...,Oil,₹164,4.0,(6),6
198,ABRO MF-390 Motor Flush(MF) Synthetic Motor Oi...,Oil,₹281,4.3,(428),428


In [12]:
df_tyres  = pd.read_csv("flipkart_tyres.csv")
df_brake  = pd.read_csv("flipkart_brake.csv")
df_clutch = pd.read_csv("flipkart_clutch.csv")
df_oil    = pd.read_csv("flipkart_oil.csv")

df_all = pd.concat([df_tyres, df_brake, df_clutch, df_oil], ignore_index=True)

df_all.to_csv("flipkart_all_products.csv", index=False)

print(f"\nCombined data saved to flipkart_all_products.csv")
print(f"Total products scraped: {len(df_all)}")
print(f"\nProducts by Category:")
print(df_all['Category'].value_counts())



Combined data saved to flipkart_all_products.csv
Total products scraped: 800

Products by Category:
Category
Tyres     200
Brake     200
Clutch    200
Oil       200
Name: count, dtype: int64


In [13]:
df_all


,Product Name,Category,Price,Rating,Reviews_Text,No_of_Reviews
0,JK Taximax 4 Wheeler Tyre,Tyres,"₹3,099",4.2,(6),6
1,JK TYRE UX Royale 4 Wheeler Tyre,Tyres,"₹4,183",3.9,(55),55
2,TVS Eurogrip ATT 52 2.75 - 18 42 P Front Two W...,Tyres,₹884,4.2,(734),734
3,CEAT 105918 Milaze TL 53J SW 90/100-10 Front &...,Tyres,"₹1,082",4.3,"(21,317)",21317
4,MRF Nylogrip FE 90/100 R10 Front & Rear Two Wh...,Tyres,"₹1,215",4.3,(370),370
...,...,...,...,...,...,...
795,Reddit 20W-40-PREMIUM4T | Smooth Ride|Better M...,Oil,₹226,4.4,(9),9
796,ESSON SUPER SYNTH 5W30 3 LTR P1 SUPER SYNTH 5W...,Oil,₹736,4.3,(964),964
797,PL SUPER PALCO 2T POWER API TC 2 Stroke Air Co...,Oil,₹164,4.0,(6),6
798,ABRO MF-390 Motor Flush(MF) Synthetic Motor Oi...,Oil,₹281,4.3,(428),428
